# Modelling: survival analysis and classification

This notebook fits four models to predict business survival in Calgary:

1. **Kaplan-Meier** estimator -- non-parametric survival curves by business type.
2. **Cox proportional hazards** -- semi-parametric model with hazard ratios.
3. **Random Forest** classifier -- binary survived / not-survived.
4. **XGBoost** classifier -- gradient-boosted binary classification.

We compare the models using C-index (survival) and AUC (classification).

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

sys.path.insert(0, '..')
from src.data_loader import load_and_preprocess
from src.model import (
    SurvivalAnalyzer,
    train_random_forest,
    train_xgboost,
    FEATURE_COLS,
    _prepare_classification_data,
    save_model,
)

pd.set_option('display.max_columns', 40)
print('Setup complete.')

## 1. Load preprocessed data

In [ ]:
licences, census = load_and_preprocess()
licences['event_observed'] = 1 - licences['survived']

print(f'Records: {len(licences):,}')
print(f'Survived: {licences["survived"].sum():,} | Closed: {licences["event_observed"].sum():,}')

## 2. Kaplan-Meier survival curves by business type

The Kaplan-Meier estimator provides a non-parametric estimate of the survival
function. We compute separate curves for the most frequent business categories
using `SurvivalAnalyzer.kaplan_meier_by_group`.

In [ ]:
analyzer = SurvivalAnalyzer()

km_curves = analyzer.kaplan_meier_by_group(
    licences,
    duration_col='business_age_days',
    event_col='event_observed',
    group_col='business_category',
    top_n=8,
)

km_curves['timeline_years'] = km_curves['timeline'] / 365.0

fig = px.line(
    km_curves, x='timeline_years', y='survival_probability', color='label',
    title='Kaplan-Meier survival curves by business type',
    labels={'timeline_years': 'Years', 'survival_probability': 'Survival probability', 'label': 'Type'},
)
fig.update_layout(height=500)
fig.show()

In [ ]:
milestones = analyzer.survival_rates_at_milestones(
    licences,
    duration_col='business_age_days',
    event_col='event_observed',
    group_col='business_category',
    milestones_years=[1, 3, 5],
    top_n=10,
)
milestones

## 3. Cox proportional hazards model

The Cox PH model estimates hazard ratios for each covariate, telling us which
features increase or decrease the hazard (risk of closure) after controlling
for other variables.

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Prepare a numeric-only dataframe for the Cox model
cox_df = licences.copy()
le = LabelEncoder()
cox_df['business_category_enc'] = le.fit_transform(cox_df['business_category'].astype(str))

cox_cols = ['business_age_days', 'event_observed', 'is_home_occupation',
            'business_count', 'business_diversity', 'avg_business_age',
            'issue_year', 'business_category_enc']
cox_data = cox_df[cox_cols].dropna()

print(f'Cox data shape: {cox_data.shape}')

In [ ]:
cph = analyzer.fit_cox(
    cox_data,
    duration_col='business_age_days',
    event_col='event_observed',
)

hazard_summary = analyzer.get_hazard_summary()
print('Cox PH concordance index:', round(cph.concordance_index_, 4))
hazard_summary[['coef', 'exp(coef)', 'p']]

In [ ]:
# Visualise hazard ratios (exp(coef))
hr = hazard_summary[['exp(coef)']].reset_index()
hr.columns = ['Feature', 'Hazard Ratio']

fig = px.bar(
    hr.sort_values('Hazard Ratio'),
    x='Hazard Ratio', y='Feature', orientation='h',
    title='Cox PH hazard ratios (exp(coef))',
    color='Hazard Ratio',
    color_continuous_scale='RdBu_r',
)
fig.add_vline(x=1.0, line_dash='dash', line_color='grey')
fig.update_layout(height=400)
fig.show()

## 4. Random Forest classifier

For a direct binary prediction (survived vs. closed) we train a Random Forest
with 200 estimators. The `train_random_forest` helper in `model.py` handles
the train/test split and metric computation.

In [ ]:
rf_result = train_random_forest(licences)

print('Random Forest metrics:')
for k, v in rf_result['metrics'].items():
    if k != 'classification_report':
        print(f'  {k}: {v}')
print(f'\n{rf_result["metrics"]["classification_report"]}')

## 5. XGBoost classifier

XGBoost often performs well on tabular data. We use the same feature set and
train/test split convention via `train_xgboost`.

In [ ]:
xgb_result = train_xgboost(licences)

print('XGBoost metrics:')
for k, v in xgb_result['metrics'].items():
    if k != 'classification_report':
        print(f'  {k}: {v}')
print(f'\n{xgb_result["metrics"]["classification_report"]}')

## 6. Compare C-index and AUC across models

The concordance index (C-index) from the Cox model and the ROC-AUC from the
classifiers both measure discriminative ability on a 0-1 scale.

In [ ]:
comparison = pd.DataFrame([
    {'Model': 'Cox PH', 'C-index / AUC': round(cph.concordance_index_, 4), 'Metric Type': 'C-index'},
    {'Model': 'Random Forest', 'C-index / AUC': rf_result['metrics']['roc_auc'], 'Metric Type': 'AUC'},
    {'Model': 'XGBoost', 'C-index / AUC': xgb_result['metrics']['roc_auc'], 'Metric Type': 'AUC'},
])

fig = px.bar(
    comparison, x='Model', y='C-index / AUC', color='Model',
    text='C-index / AUC',
    title='Model comparison: C-index and AUC',
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.update_traces(textposition='outside', texttemplate='%{text:.4f}')
fig.update_layout(height=400, yaxis_range=[0, 1], showlegend=False)
fig.show()

comparison

In [ ]:
# Full classification comparison
clf_comparison = pd.DataFrame([
    {'Model': 'Random Forest', **{k: v for k, v in rf_result['metrics'].items() if k != 'classification_report'}},
    {'Model': 'XGBoost', **{k: v for k, v in xgb_result['metrics'].items() if k != 'classification_report'}},
])
clf_comparison

## 7. Save best model

Persist the better-performing classifier for use by the Streamlit app.

In [ ]:
best_name = 'XGBoost' if xgb_result['metrics']['roc_auc'] >= rf_result['metrics']['roc_auc'] else 'Random Forest'
best_result = xgb_result if best_name == 'XGBoost' else rf_result

path = save_model(best_result['model'], f'{best_name.lower().replace(" ", "_")}_survival.joblib')
print(f'Saved {best_name} model to {path}')

## Summary

- Kaplan-Meier curves show clear differences in survival trajectories across business types.
- The Cox PH model yields interpretable hazard ratios with a C-index around 0.68.
- Both Random Forest and XGBoost achieve reasonable AUC for binary survival prediction.
- The next notebook (`04_evaluation.ipynb`) digs deeper into per-segment results,
  feature importance, and location recommendations.